---
title: "Lab: Data Quality Gates, Great Expectations, and OpenLineage on Real Retail Data"
format: html
---

# Day 4 Lab — Data Quality, Governance, and Lineage
## Using Real-World E-Commerce Data (Kaggle)

We are using the **UCI Online Retail Dataset** — 541,909 real transactions from a UK-based online retailer (2010–2011).
This dataset has genuine quality problems: 25% of CustomerIDs are missing, cancelled orders have negative quantities, and there are formatting inconsistencies.

Your pipeline will:
1. Download the dataset from Kaggle
2. Profile the raw data to discover quality issues
3. Validate records against a Pydantic v2 data contract
4. Run all 6 DAMA quality dimensions on the clean subset
5. Route data to production or quarantine based on quality verdict
6. Run the same checks again through a **real Great Expectations checkpoint**
7. Emit an OpenLineage-style audit trail, and also **real openlineage-python** events


## Step 1 — Install Dependencies

In [1]:
!pip install kagglehub pandas loguru "pydantic>=2.0" pyarrow great_expectations openlineage-python


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 1.7 MB/s eta 0:00:00


## Step 2 — Kaggle Authentication

You need a Kaggle account and API key.

**Option A — Colab Secrets (recommended):**
1. Go to [kaggle.com](https://www.kaggle.com) → Account → API → Create New Token → downloads `kaggle.json`
2. In Colab: click the 🔑 icon (left sidebar) → add two secrets:
   - `KAGGLE_USERNAME` = your Kaggle username
   - `KAGGLE_KEY` = the key value from `kaggle.json`

**Option B — Upload `kaggle.json` directly:**
Run the upload cell below instead.

In [2]:
# ── Option A: Colab Secrets ──────────────────────────────────────────
import os

try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")
    print("✅ Kaggle credentials loaded from Colab Secrets.")
except Exception:
    print("⚠️  Colab Secrets not available — run Option B below.")

✅ Kaggle credentials loaded from Colab Secrets.


In [3]:
# ── Option B: Upload kaggle.json manually (skip if Option A worked) ──
# from google.colab import files
# uploaded = files.upload()          # select your kaggle.json
# import os, shutil
# os.makedirs("/root/.kaggle", exist_ok=True)
# shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
# os.chmod("/root/.kaggle/kaggle.json", 0o600)
# print("✅ kaggle.json installed.")

## Step 3 — Download the Dataset

In [4]:
import kagglehub, glob

# UCI Online Retail Dataset — 541,909 real transactions, ~45 MB
path = kagglehub.dataset_download("carrie1/ecommerce-data")
print(f"Downloaded to: {path}")

csv_files = glob.glob(f"{path}/**/*.csv", recursive=True)
print(f"CSV files found: {csv_files}")
RAW_CSV = csv_files[0]
print(f"Using: {RAW_CSV}")

100%|██████████| 7.20M/7.20M [00:00<00:00, 115MB/s]

Extracting files...


Downloaded to: /root/.cache/kagglehub/datasets/carrie1/ecommerce-data/versions/1
CSV files found: ['/root/.cache/kagglehub/datasets/carrie1/ecommerce-data/versions/1/data.csv']
Using: /root/.cache/kagglehub/datasets/carrie1/ecommerce-data/versions/1/data.csv


## Step 4 — Load and Profile the Raw Data

Before running any pipeline, always **profile** the data to understand what you are dealing with.
This is the discovery phase — you are looking for quality issues before writing any validation rules.

In [5]:
import pandas as pd
from datetime import datetime

df_raw = pd.read_csv(RAW_CSV, encoding="ISO-8859-1", dtype=str)
print(f"Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head(5)

Shape: 541,909 rows × 8 columns
Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850,United Kingdom


In [6]:
# ── Data profiling ───────────────────────────────────────────────────
print("=" * 55)
print("  RAW DATA QUALITY PROFILE")
print("=" * 55)

# Null counts per column
nulls = df_raw.isnull().sum()
print("\nNull counts per column:")
for col, n in nulls.items():
    pct = 100 * n / len(df_raw)
    bar = "█" * int(pct / 5)
    print(f"  {col:<15} {n:>7,}  ({pct:5.1f}%)  {bar}")

# Basic type conversion to inspect value ranges
df_raw["Quantity"]  = pd.to_numeric(df_raw["Quantity"],  errors="coerce")
df_raw["UnitPrice"] = pd.to_numeric(df_raw["UnitPrice"], errors="coerce")

print(f"\nQuantity range  : {df_raw['Quantity'].min():,.0f}  →  {df_raw['Quantity'].max():,.0f}")
print(f"  Negative (cancellations): {(df_raw['Quantity'] < 0).sum():,} rows")
print(f"  Zero quantity           : {(df_raw['Quantity'] == 0).sum():,} rows")

print(f"\nUnitPrice range : {df_raw['UnitPrice'].min():.2f}  →  {df_raw['UnitPrice'].max():.2f}")
print(f"  Negative price          : {(df_raw['UnitPrice'] < 0).sum():,} rows")
print(f"  Zero price              : {(df_raw['UnitPrice'] == 0).sum():,} rows")

dupes = df_raw.duplicated().sum()
print(f"\nDuplicate rows  : {dupes:,}")

countries = df_raw["Country"].nunique()
print(f"\nUnique countries: {countries}")
print(f"  Sample: {list(df_raw['Country'].unique()[:8])}")

print("=" * 55)
print("\n💡 Key findings:")
print(f"   • CustomerID is NULL in {nulls.get('CustomerID', 0):,} rows ({100*nulls.get('CustomerID',0)/len(df_raw):.1f}%)")
print(f"   • {(df_raw['Quantity'] < 0).sum():,} rows have negative Quantity (cancelled orders)")
print(f"   • {(df_raw['UnitPrice'] == 0).sum():,} rows have UnitPrice = 0 (no-charge items)")
print(f"   • {dupes:,} exact duplicate rows detected")

  RAW DATA QUALITY PROFILE

Null counts per column:
  InvoiceNo             0  (  0.0%)  
  StockCode             0  (  0.0%)  
  Description       1,454  (  0.3%)  
  Quantity              0  (  0.0%)  
  InvoiceDate           0  (  0.0%)  
  UnitPrice             0  (  0.0%)  
  CustomerID      135,080  ( 24.9%)  ████
  Country               0  (  0.0%)  

Quantity range  : -80,995  →  80,995
  Negative (cancellations): 10,624 rows
  Zero quantity           : 0 rows

UnitPrice range : -11062.06  →  38970.00
  Negative price          : 2 rows
  Zero price              : 2,515 rows

Duplicate rows  : 5,268

Unique countries: 38
  Sample: ['United Kingdom', 'France', 'Australia', 'Netherlands', 'Germany', 'Norway', 'EIRE', 'Switzerland']

💡 Key findings:
   • CustomerID is NULL in 135,080 rows (24.9%)
   • 10,624 rows have negative Quantity (cancelled orders)
   • 2,515 rows have UnitPrice = 0 (no-charge items)
   • 5,268 exact duplicate rows detected


## Step 5 — Data Contract (Pydantic v2)

Now that we know the quality issues, we define a **contract** at the pipeline boundary.
Any record that violates the contract is rejected immediately — before it enters the pipeline.

In [7]:
import re
from pydantic import BaseModel, field_validator, ConfigDict

class RetailTransactionContract(BaseModel):
    """
    Machine-enforceable schema for the Online Retail dataset.
    strict=True means no silent type coercion — '123' stays a string.
    """
    model_config = ConfigDict(strict=False)   # lenient for CSV strings

    InvoiceNo:   str
    StockCode:   str
    Quantity:    float
    UnitPrice:   float
    CustomerID:  str
    Country:     str

    @field_validator("CustomerID")
    @classmethod
    def customer_id_required(cls, v: str) -> str:
        if not v or v.strip() in ("", "nan", "None"):
            raise ValueError("CustomerID is required — cannot be null")
        return v.strip()

    @field_validator("Quantity")
    @classmethod
    def positive_quantity(cls, v: float) -> float:
        if v <= 0:
            raise ValueError(f"Quantity must be > 0 (got {v} — likely a cancellation)")
        return v

    @field_validator("UnitPrice")
    @classmethod
    def positive_price(cls, v: float) -> float:
        if v <= 0:
            raise ValueError(f"UnitPrice must be > 0 (got {v})")
        return v

    @field_validator("InvoiceNo")
    @classmethod
    def valid_invoice(cls, v: str) -> str:
        if not re.match(r"^[A-Z]?\d{5,6}$", v.strip()):
            raise ValueError(f"InvoiceNo format invalid: '{v}'")
        return v.strip()

print("✅ RetailTransactionContract defined.")

✅ RetailTransactionContract defined.


In [8]:
import pandas as pd
# ── Apply contract validation ──────────────────────────────────────────────
from loguru import logger
logger.add("pipeline_execution.log", format="{time:YYYY-MM-DD HH:mm:ss} | {level} | {message}")

print("Running contract validation on all rows (this may take ~30 seconds)...")

valid_rows, rejected_rows = [], []

for _, row in df_raw.iterrows():
    try:
        RetailTransactionContract(
            InvoiceNo  = str(row.get("InvoiceNo",  "") or ""),
            StockCode  = str(row.get("StockCode",  "") or ""),
            Quantity   = row.get("Quantity",  0),
            UnitPrice  = row.get("UnitPrice", 0),
            CustomerID = str(row.get("CustomerID", "") or ""),
            Country    = str(row.get("Country",    "") or ""),
        )
        valid_rows.append(row)
    except Exception as exc:
        row_copy = dict(row)
        row_copy["rejection_reason"] = str(exc)
        rejected_rows.append(row_copy)

df_valid    = pd.DataFrame(valid_rows)
df_rejected = pd.DataFrame(rejected_rows)

total = len(df_raw)
print(f"\n{'='*55}")
print(f"  CONTRACT VALIDATION RESULTS")
print(f"{'='*55}")
print(f"  Total rows    : {total:>10,}")
print(f"  ✅ Valid       : {len(df_valid):>10,}  ({100*len(df_valid)/total:.1f}%)")
print(f"  ❌ Rejected    : {len(df_rejected):>10,}  ({100*len(df_rejected)/total:.1f}%)")
print(f"{'='*55}")

if not df_rejected.empty:
    reasons = pd.DataFrame(df_rejected)["rejection_reason"]
    print("\nTop rejection reasons:")
    print(reasons.str.split(":").str[0].value_counts().head(5).to_string())

Running contract validation on all rows (this may take ~30 seconds)...

  CONTRACT VALIDATION RESULTS
  Total rows    :    541,909
  ✅ Valid       :    397,884  (73.4%)
  ❌ Rejected    :    144,025  (26.6%)

Top rejection reasons:
rejection_reason
1 validation error for RetailTransactionContract\nCustomerID\n  Value error, CustomerID is required — cannot be null [type=value_error, input_value='nan', input_type=str]\n    For further information visit https               132220
1 validation error for RetailTransactionContract\nQuantity\n  Value error, Quantity must be > 0 (got -1.0 — likely a cancellation) [type=value_error, input_value=-1, input_type=int]\n    For further information visit https      3848
1 validation error for RetailTransactionContract\nQuantity\n  Value error, Quantity must be > 0 (got -2.0 — likely a cancellation) [type=value_error, input_value=-2, input_type=int]\n    For further information visit https      1329
2 validation errors for RetailTransactionContract\nUn

## Step 6 — DAMA 6-Dimension Quality Engine

We run all six DAMA quality dimensions on the **contract-validated** subset.
These are not planted anomalies — they are real patterns in the data.

In [9]:
import json, uuid
from datetime import timedelta, datetime, UTC

class DataQualityEngine:
    """
    Evaluates 6 DAMA dimensions on the Online Retail data:
      1. Completeness  — no nulls in critical fields
      2. Accuracy      — prices and quantities in valid ranges
      3. Consistency   — same InvoiceNo maps to same Country
      4. Timeliness    — all invoices within expected date range
      5. Uniqueness    — no full row duplicates
      6. Validity      — StockCode matches expected format
    """
    EXPECTED_START = pd.Timestamp("2010-01-01")
    EXPECTED_END   = pd.Timestamp("2012-01-01")

    def __init__(self, df: pd.DataFrame):
        self.df  = df.copy()
        self.log = {}

    def run(self) -> dict:
        logger.info("Starting DAMA 6-dimension scan on real retail data...")

        # 1. Completeness — Description field (optional but check anyway)
        null_desc = self.df["Description"].isnull().sum() if "Description" in self.df.columns else 0
        self._record("1_completeness_description",
                     null_desc == 0,
                     f"{null_desc:,} rows missing Description")

        # 2. Accuracy — UnitPrice and Quantity in business range
        high_price = (self.df["UnitPrice"] > 10_000).sum()
        self._record("2_accuracy_unit_price",
                     high_price == 0,
                     f"{high_price:,} rows with UnitPrice > £10,000 (likely data entry error)")

        # 3. Consistency — same InvoiceNo should map to same Country
        inconsistent = (
            self.df.groupby("InvoiceNo")["Country"].nunique().max() > 1
        )
        self._record("3_consistency_invoice_country",
                     not inconsistent,
                     "Some InvoiceNos have transactions from multiple countries")

        # 4. Timeliness — InvoiceDate within expected range
        if "InvoiceDate" in self.df.columns:
            try:
                dates = pd.to_datetime(self.df["InvoiceDate"], errors="coerce")
                out_of_range = ((dates < self.EXPECTED_START) | (dates > self.EXPECTED_END)).sum()
                self._record("4_timeliness_invoice_date",
                             out_of_range == 0,
                             f"{out_of_range:,} invoices outside 2010–2012 range")
            except Exception:
                self._record("4_timeliness_invoice_date", False, "Could not parse InvoiceDate")

        # 5. Uniqueness — no fully duplicated rows
        dupes = self.df.duplicated().sum()
        self._record("5_uniqueness_rows",
                     dupes == 0,
                     f"{dupes:,} exact duplicate rows")

        # 6. Validity — StockCode should be 5 digits or 5 digits + letter
        invalid_sc = (~self.df["StockCode"].astype(str).str.match(
            r"^\d{5}[A-Z]?$", na=False
        )).sum()
        self._record("6_validity_stock_code",
                     invalid_sc == 0,
                     f"{invalid_sc:,} StockCodes don't match 5-digit format")

        passed = all(v["passed"] for v in self.log.values())
        return {"passed": passed, "dimensions": self.log,
                "timestamp": datetime.now(UTC).isoformat(), "row_count": len(self.df)}

    def _record(self, name, passed, detail):
        self.log[name] = {"passed": passed,
                          "status": "PASSED" if passed else "FAILED",
                          "detail": detail}
        if passed:
            logger.success(f"  [PASSED] {name}: {detail}")
        else:
            logger.warning(f"  [FAILED] {name}: {detail}")

# Run the engine
dq = DataQualityEngine(df_valid)
report = dq.run()

print("\n" + "="*55)
print("  DAMA DATA QUALITY REPORT")
print("="*55)
for dim, result in report["dimensions"].items():
    icon = "✅" if result["passed"] else "❌"
    print(f"  {icon}  {dim}")
    print(f"       {result['detail']}")
print("="*55)
print(f"\nOverall verdict: {'PASSED ✅' if report['passed'] else 'FAILED ❌'}")
print(f"Rows evaluated : {report['row_count']:,}")

2026-07-01 11:01:02.194 | INFO     | __main__:run:22 - Starting DAMA 6-dimension scan on real retail data...
2026-07-01 11:01:02.218 | SUCCESS  | __main__:_record:78 -   [PASSED] 1_completeness_description: 0 rows missing Description
2026-07-01 11:01:02.222 | SUCCESS  | __main__:_record:78 -   [PASSED] 2_accuracy_unit_price: 0 rows with UnitPrice > £10,000 (likely data entry error)
2026-07-01 11:01:02.299 | SUCCESS  | __main__:_record:78 -   [PASSED] 3_consistency_invoice_country: Some InvoiceNos have transactions from multiple countries
2026-07-01 11:01:02.416 | SUCCESS  | __main__:_record:78 -   [PASSED] 4_timeliness_invoice_date: 0 invoices outside 2010–2012 range
2026-07-01 11:01:02.695 | WARNING  | __main__:_record:80 -   [FAILED] 5_uniqueness_rows: 5,192 exact duplicate rows
2026-07-01 11:01:02.886 | WARNING  | __main__:_record:80 -   [FAILED] 6_validity_stock_code: 1,838 StockCodes don't match 5-digit format



  DAMA DATA QUALITY REPORT
  ✅  1_completeness_description
       0 rows missing Description
  ✅  2_accuracy_unit_price
       0 rows with UnitPrice > £10,000 (likely data entry error)
  ✅  3_consistency_invoice_country
       Some InvoiceNos have transactions from multiple countries
  ✅  4_timeliness_invoice_date
       0 invoices outside 2010–2012 range
  ❌  5_uniqueness_rows
       5,192 exact duplicate rows
  ❌  6_validity_stock_code
       1,838 StockCodes don't match 5-digit format

Overall verdict: FAILED ❌
Rows evaluated : 397,884


## Step 6b — Real Great Expectations Checkpoint

The DAMA scan above is a hand-rolled implementation of the same six dimensions. Here we run
the core checks again through a **real Great Expectations 1.x checkpoint** — the actual
library the capstone rubric names — so you see both the concept and the real tool.


In [10]:
import great_expectations as gx
import great_expectations.expectations as gxe

def run_great_expectations_checkpoint(df):
    """Runs core quality rules as a real GX 1.x fluent-API checkpoint."""
    context = gx.get_context(mode="ephemeral")
    data_source = context.data_sources.add_pandas("pandas_dq_lab")
    data_asset = data_source.add_dataframe_asset(name="retail_transactions")
    batch_definition = data_asset.add_batch_definition_whole_dataframe("whole_df")

    suite = context.suites.add(gx.ExpectationSuite(name="retail_quality_suite"))
    suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column="CustomerID"))
    suite.add_expectation(gxe.ExpectColumnValuesToBeBetween(column="Quantity", min_value=0.0001))
    suite.add_expectation(gxe.ExpectColumnValuesToBeBetween(column="UnitPrice", min_value=0.0001))
    suite.add_expectation(gxe.ExpectColumnValuesToMatchRegex(column="InvoiceNo", regex=r"^[A-Z]?\d{5,6}$"))

    validation_definition = context.validation_definitions.add(
        gx.ValidationDefinition(
            name="retail_quality_validation",
            data=batch_definition,
            suite=suite,
        )
    )
    checkpoint = context.checkpoints.add(
        gx.Checkpoint(
            name="retail_quality_checkpoint",
            validation_definitions=[validation_definition],
        )
    )
    result = checkpoint.run(batch_parameters={"dataframe": df})

    print(f"[GX] Real Great Expectations checkpoint success={result.success}")
    for run_result in result.run_results.values():
        for r in run_result["results"]:
            status = "PASSED" if r["success"] else "FAILED"
            print(f"  [GX] {status} {r['expectation_config']['type']}")
    return result.success


gx_passed = run_great_expectations_checkpoint(df_valid)
print(f"\n[GX] Real Great Expectations checkpoint agrees: success={gx_passed}")


[GX] Real Great Expectations checkpoint success=True
  [GX] PASSED expect_column_values_to_not_be_null
  [GX] PASSED expect_column_values_to_be_between
  [GX] PASSED expect_column_values_to_be_between
  [GX] PASSED expect_column_values_to_match_regex

[GX] Real Great Expectations checkpoint agrees: success=True


## Step 7 — Route: Production Warehouse or Quarantine

Based on the quality verdict, we either promote the data or quarantine it.
Regardless of the verdict, we always write the rejected rows to quarantine
so producers can fix them.

In [11]:
import os
from datetime import datetime, UTC

os.makedirs("production_warehouse", exist_ok=True)
os.makedirs("quarantine_zone",      exist_ok=True)

# Always quarantine contract-rejected rows
if not df_rejected.empty:
    q_path = f"quarantine_zone/contract_violations_{int(datetime.now(UTC).timestamp())}.csv"
    pd.DataFrame(df_rejected).to_csv(q_path, index=False)
    logger.error(f"{len(df_rejected):,} contract violations → {q_path}")

# Route based on DAMA verdict
if report["passed"]:
    out = "production_warehouse/online_retail_clean.parquet"
    df_valid.to_parquet(out, index=False)
    logger.success(f"Quality gate PASSED — {len(df_valid):,} rows → {out}")
    print(f"\n✅ Production write: {out}")
    print(f"   {len(df_valid):,} clean rows ready for downstream analytics")
else:
    q_path = f"quarantine_zone/dq_failure_{int(datetime.now(UTC).timestamp())}.csv"
    df_valid.to_csv(q_path, index=False)
    logger.error(f"Quality gate FAILED — {len(df_valid):,} rows → {q_path}")
    print(f"\n❌ Quality gate failed — data quarantined to {q_path}")
    print("   Fix the flagged dimensions and re-run the pipeline.")

2026-07-01 11:01:05.556 | ERROR    | __main__:<cell line: 0>:11 - 144,025 contract violations → quarantine_zone/contract_violations_1782903663.csv
2026-07-01 11:01:07.441 | ERROR    | __main__:<cell line: 0>:23 - Quality gate FAILED — 397,884 rows → quarantine_zone/dq_failure_1782903665.csv



❌ Quality gate failed — data quarantined to quarantine_zone/dq_failure_1782903665.csv
   Fix the flagged dimensions and re-run the pipeline.


## Step 8 — OpenLineage Audit Trail

Every pipeline run emits structured lineage events.
In production these would be sent to a **Marquez** server via HTTP POST.
Here we write them locally for inspection.

In [12]:
import json, uuid, os
from datetime import datetime, UTC

class LineageEmitter:
    def __init__(self, job_name):
        self.job_name = job_name
        self.run_id   = str(uuid.uuid4())
        self.events   = []

    def emit(self, event_type, input_ds, output_ds, facets=None):
        self.events.append({
            "eventType": event_type,
            "eventTime": datetime.now(UTC).isoformat(),
            "run":    {"runId": self.run_id},
            "job":    {"namespace": "day4_lab", "name": self.job_name},
            "inputs":  [{"namespace": "kaggle", "name": input_ds}],
            "outputs": [{"namespace": "lab",    "name": output_ds,
                         "facets": facets or {}}],
        })
        print(f"[LINEAGE] {event_type:10s} | {input_ds} → {output_ds}")

    def write(self, path="lineage_events/run.json"):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, "w") as f:
            json.dump(self.events, f, indent=2)
        print(f"\n📋 {len(self.events)} lineage events → {path}")


lineage = LineageEmitter("retail_quality_pipeline")

lineage.emit("START",    "online_retail_raw.csv", "landing_zone",
             {"row_count": len(df_raw)})

lineage.emit("COMPLETE" if df_rejected.empty else "FAIL",
             "landing_zone", "quarantine_zone/contract_violations",
             {"rejected_rows": len(df_rejected),
              "rejection_rate_pct": round(100 * len(df_rejected) / len(df_raw), 2)})

if report["passed"]:
    lineage.emit("COMPLETE", "landing_zone", "production_warehouse/online_retail_clean",
                 {"row_count": len(df_valid),
                  "dama_dimensions_passed": sum(1 for v in report["dimensions"].values() if v["passed"]),
                  "dama_dimensions_total":  len(report["dimensions"])})
else:
    failed_dims = [k for k, v in report["dimensions"].items() if not v["passed"]]
    lineage.emit("FAIL", "landing_zone", "quarantine_zone/dq_failure",
                 {"row_count": len(df_valid), "failed_dimensions": failed_dims})

lineage.write()

[LINEAGE] START      | online_retail_raw.csv → landing_zone
[LINEAGE] FAIL       | landing_zone → quarantine_zone/contract_violations
[LINEAGE] FAIL       | landing_zone → quarantine_zone/dq_failure

📋 3 lineage events → lineage_events/run.json


## Step 8b — Real OpenLineage Emitter

`LineageEmitter` above hand-builds JSON matching the OpenLineage event shape. Here we emit
**real openlineage-python** START/COMPLETE (or FAIL) events to a local file transport — no
Marquez server required — as the real-library counterpart.


In [13]:
from openlineage.client import OpenLineageClient
from openlineage.client.transport.file import FileConfig, FileTransport
from openlineage.client.event_v2 import RunEvent, RunState, Run, Job
from openlineage.client.uuid import generate_new_uuid

def emit_real_openlineage_events(job_name, row_count, passed):
    """Emits real OpenLineage START/COMPLETE (or FAIL) events to a local file transport."""
    os.makedirs("lineage_events", exist_ok=True)
    transport = FileTransport(FileConfig(log_file_path="lineage_events/openlineage_run.log"))
    client = OpenLineageClient(transport=transport)

    run_id = str(generate_new_uuid())
    job = Job(namespace="day4_lab", name=job_name)
    run = Run(runId=run_id)
    now = datetime.now(UTC).isoformat()

    client.emit(RunEvent(
        eventType=RunState.START, eventTime=now, run=run, job=job,
        producer="https://github.com/sdaia/modern-data-engineering-lab",
    ))
    client.emit(RunEvent(
        eventType=RunState.COMPLETE if passed else RunState.FAIL,
        eventTime=now, run=run, job=job,
        producer="https://github.com/sdaia/modern-data-engineering-lab",
    ))
    print(f"[OpenLineage] real START/{'COMPLETE' if passed else 'FAIL'} events "
          f"({row_count} rows) -> lineage_events/openlineage_run.log*")


emit_real_openlineage_events("retail_quality_pipeline", len(df_valid), report["passed"])


[OpenLineage] real START/FAIL events (397884 rows) -> lineage_events/openlineage_run.log*


## Step 9 — Explore the Clean Data

Now let's actually look at what passed the quality gates and draw some insights.

In [14]:
if report["passed"] and os.path.exists("production_warehouse/online_retail_clean.parquet"):
    df_clean = pd.read_parquet("production_warehouse/online_retail_clean.parquet")
else:
    df_clean = df_valid.copy()

df_clean["Revenue"] = df_clean["Quantity"].astype(float) * df_clean["UnitPrice"].astype(float)

print("📊 Top 10 countries by revenue (clean data only):")
top_countries = (
    df_clean.groupby("Country")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
print(top_countries.apply(lambda x: f"  £{x:>12,.2f}").to_string())

print(f"\n📦 Unique products  : {df_clean['StockCode'].nunique():,}")
print(f"👥 Unique customers : {df_clean['CustomerID'].nunique():,}")
print(f"🧾 Unique invoices  : {df_clean['InvoiceNo'].nunique():,}")
print(f"💷 Total revenue    : £{df_clean['Revenue'].sum():,.2f}")

print("\n✅ Pipeline complete.")
print(f"   Started with {len(df_raw):,} raw rows.")
print(f"   {len(df_rejected):,} rejected by contract ({100*len(df_rejected)/len(df_raw):.1f}%).")
print(f"   {len(df_clean):,} rows in production warehouse ready for analytics.")

📊 Top 10 countries by revenue (clean data only):
Country
United Kingdom      £7,308,391.55
Netherlands         £  285,446.34
EIRE                £  265,545.90
Germany             £  228,867.14
France              £  209,024.05
Australia           £  138,521.31
Spain               £   61,577.11
Switzerland         £   56,443.95
Belgium             £   41,196.34
Sweden              £   38,378.33

📦 Unique products  : 3,665
👥 Unique customers : 4,338
🧾 Unique invoices  : 18,532
💷 Total revenue    : £8,911,407.90

✅ Pipeline complete.
   Started with 541,909 raw rows.
   144,025 rejected by contract (26.6%).
   397,884 rows in production warehouse ready for analytics.
